In [1]:
import pandas as pd
from IPython.display import display, HTML
import evaluate
from rouge_score import rouge_scorer
from pathlib import Path

c:\Users\erhon\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Define o caminho do arquivo
path_baseline = Path("data/mediasum") / "mediasum_amostras_baseline_avaliacao_humana.csv"
path_ft = Path("data/mediasum") / "mediasum_amostras_finetuning_avaliacao_humana.csv"
path_ft_results = Path("data/mediasum") / "mediasum_resultados_finetuning.csv"


In [3]:
def alinhar_amostras(caminho_baseline, caminho_ft_completo, caminho_ft_saida, campo_texto="dialogo"):
    """
    Alinha as 40 amostras do FT com as do baseline usando resumo_referencia como chave.
    Mantém o rouge_l_individual calculado para cada um separadamente.
    Salva o CSV corrigido do FT na mesma ordem do baseline.
    
    Parâmetros:
    - caminho_baseline: CSV das 40 amostras do baseline
    - caminho_ft_completo: CSV com todos os resumos gerados pelo FT (resultados_finetuning.csv)
    - caminho_ft_saida: caminho onde salvar o CSV corrigido do FT
    - campo_texto: "dialogo" para SAMSum/MediaSum, "input_texto" para QMSum
    """
    df_base = pd.read_csv(caminho_baseline)
    df_ft = pd.read_csv(caminho_ft_completo)

    # Alinha pelo resumo_referencia mantendo a ordem do baseline
    df_ft_alinhado = df_base[["resumo_referencia"]].merge(
        df_ft,
        on="resumo_referencia",
        how="left"
    )

    # Verifica se todas as 40 foram encontradas
    nao_encontradas = df_ft_alinhado["resumo_gerado"].isna().sum()
    if nao_encontradas > 0:
        print(f"ATENÇÃO: {nao_encontradas} amostras não encontradas no FT!")
        return None

    # Calcula ROUGE-L individual
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

    def calcular_rouge_l(row):
        scores = scorer.score(row["resumo_referencia"], row["resumo_gerado"])
        return scores["rougeL"].fmeasure

    print("Calculando ROUGE-L individual do FT...")
    df_ft_alinhado["rouge_l_individual"] = df_ft_alinhado.apply(calcular_rouge_l, axis=1)

    # Adiciona colunas vazias para avaliação humana e G-Eval
    for col in ["human_fluency", "human_coherence", "human_consistency", "human_relevance",
                "geval_fluency", "geval_coherence", "geval_consistency", "geval_relevance"]:
        df_ft_alinhado[col] = ""

    # Reordena colunas igual ao baseline
    df_ft_alinhado = df_ft_alinhado[df_base.columns.tolist()]

    # Salva o CSV corrigido
    # df_ft_alinhado.to_csv(caminho_ft_saida, sep=';', index=False)
    df_ft_alinhado.to_csv(caminho_ft_saida, index=False)

    # Confirmação final
    df_check = pd.read_csv(caminho_ft_saida)
    erros = []
    for i in range(len(df_base)):
        if df_base.iloc[i]["resumo_referencia"] != df_check.iloc[i]["resumo_referencia"]:
            erros.append(i+1)

    if len(erros) == 0:
        print("Todas as 40 amostras estão alinhadas corretamente!")
    else:
        print(f"ATENÇÃO: Desalinhamentos nas posições: {erros}")
        return None

    print(f"CSV do FT corrigido salvo em: {caminho_ft_saida}")
    print(f"\nROUGE-L individual — Baseline (primeiras 5):")
    print(df_base["rouge_l_individual"].head().values)
    print(f"\nROUGE-L individual — FT (primeiras 5):")
    print(df_check["rouge_l_individual"].head().values)

    return df_ft_alinhado

In [4]:
# =======================================================================
# USO — altere os caminhos conforme o dataset
# =======================================================================

# MediaSum
df_ft_mediasum = alinhar_amostras(
    caminho_baseline=path_baseline,
    caminho_ft_completo=path_ft_results,
    caminho_ft_saida=path_ft,
    campo_texto="resumo_referencia"
)

Calculando ROUGE-L individual do FT...
Todas as 40 amostras estão alinhadas corretamente!
CSV do FT corrigido salvo em: data\mediasum\mediasum_amostras_finetuning_avaliacao_humana.csv

ROUGE-L individual — Baseline (primeiras 5):
[0.10526316 0.125      0.15151515 0.42105263 0.        ]

ROUGE-L individual — FT (primeiras 5):
[0.09836066 0.06666667 0.12903226 0.12121212 0.05      ]


In [5]:
# Confirmação final do alinhamento entre baseline e FT corrigido
df_base = pd.read_csv(path_baseline)
df_ft = pd.read_csv(path_ft)

print(f"Linhas baseline: {len(df_base)}")
print(f"Linhas FT:       {len(df_ft)}")

erros = []
for i in range(len(df_base)):
    if df_base.iloc[i]["dialogo"] != df_ft.iloc[i]["dialogo"]:
        erros.append(i+1)

if len(erros) == 0:
    print("\nTodas as 40 amostras estão alinhadas corretamente!")
else:
    print(f"\nDesalinhamentos nas posições: {erros}")

print("\n--- Verificação visual (primeiras 5 amostras) ---")
for i in range(5):
    match = "✓" if df_base.iloc[i]["dialogo"] == df_ft.iloc[i]["dialogo"] else "✗"
    print(f"\nAmostra {i+1} {match}")
    print(f"Baseline: {df_base.iloc[i]['dialogo']}")
    print(f"FT:       {df_ft.iloc[i]['dialogo']}")

Linhas baseline: 40
Linhas FT:       40

Todas as 40 amostras estão alinhadas corretamente!

--- Verificação visual (primeiras 5 amostras) ---

Amostra 1 ✓
Baseline: LEMON: Some breaking news right now on the tapes that Donald Trump and his former fixer, Michael Cohen. To tell you about -- The Washington Post reporting the feds have received more than 100 recordings. So joining now to discuss, CNN Political Analyst Carl Bernstein and CNN Political Analyst Laura Coates, lots to discuss, so Laura, let's get right to it. Good evening by the way. The Washington Post is saying that the most substantive exchange involving the President is the one that was released by Cohen's attorney on Tuesday. But what is the danger for Trump in these other tapes?
LAURA COATES, POLITICAL ANALYST, CNN: Well, remember. The danger here is whether or not those conversations are going to be privileged. Just because you speak to a lawyer does not actually mean it's going to have the attorney-client privilege sta

In [6]:
df_base.to_csv('data/mediasum/mediasum_baseline_evaluate.csv', sep=';', index=False)
df_ft.to_csv('data/mediasum/mediasum_ft_evaluate.csv', sep=';', index=False)

In [7]:
def visualizar_comparativo(caminho_baseline, caminho_ft, dataset="samsum"):
    df_base = pd.read_csv(caminho_baseline)
    df_ft = pd.read_csv(caminho_ft)

    if dataset == "qmsum":
        campo_texto = "input_texto"
        label_texto = "Transcrição + Query"
    elif dataset == "mediasum":
        campo_texto = "dialogo"
        label_texto = "Transcrição"
    else:
        campo_texto = "dialogo"
        label_texto = "Diálogo"

    print(f"Total de amostras: {len(df_base)}")
    print(f"Dataset: {dataset.upper()}")
    print("="*80)

    for i in range(len(df_base)):
        row_base = df_base.iloc[i]
        row_ft = df_ft.iloc[i]

        rouge_base = round(row_base.get("rouge_l_individual", 0), 4)
        rouge_ft = round(row_ft.get("rouge_l_individual", 0), 4)

        def faixa(rouge, df):
            if rouge < df["rouge_l_individual"].quantile(0.33):
                return "BAIXO"
            elif rouge < df["rouge_l_individual"].quantile(0.66):
                return "MÉDIO"
            return "ALTO"

        def badge(rouge, df):
            f = faixa(rouge, df)
            bg = {'ALTO': '#c8e6c9', 'MÉDIO': '#fff9c4', 'BAIXO': '#ffcdd2'}[f]
            cor = {'ALTO': '#1b5e20', 'MÉDIO': '#e65100', 'BAIXO': '#b71c1c'}[f]
            return f'<span style="background:{bg};color:{cor};padding:3px 8px;border-radius:10px;font-size:11px;font-weight:600;">ROUGE-L: {rouge} — {f}</span>'

        html = f"""
        <div style="
            border: 1px solid #ccc;
            border-radius: 8px;
            padding: 16px;
            margin-bottom: 24px;
            font-family: sans-serif;
            background: #ffffff;
        ">
            <div style="
                display: flex;
                justify-content: space-between;
                align-items: center;
                margin-bottom: 14px;
            ">
                <span style="font-size: 16px; font-weight: 600; color: #111111;">
                    Amostra {i+1} de {len(df_base)}
                </span>
            </div>

            <div style="margin-bottom: 12px;">
                <div style="
                    font-size: 11px;
                    font-weight: 700;
                    text-transform: uppercase;
                    color: #444444;
                    margin-bottom: 4px;
                ">{label_texto}</div>
                <div style="
                    background: #f0f0f0;
                    border-radius: 4px;
                    padding: 10px;
                    font-size: 13px;
                    line-height: 1.6;
                    color: #111111;
                    max-height: 200px;
                    overflow-y: auto;
                    white-space: pre-wrap;
                ">{row_base[campo_texto]}</div>
            </div>

            <div style="margin-bottom: 12px;">
                <div style="
                    font-size: 11px;
                    font-weight: 700;
                    text-transform: uppercase;
                    color: #444444;
                    margin-bottom: 4px;
                ">Resumo de referência</div>
                <div style="
                    background: #c8e6c9;
                    border-radius: 4px;
                    padding: 10px;
                    font-size: 13px;
                    line-height: 1.6;
                    color: #1b5e20;
                    white-space: pre-wrap;
                ">{row_base["resumo_referencia"]}</div>
            </div>

            <div style="margin-bottom: 12px;">
                <div style="
                    display: flex;
                    justify-content: space-between;
                    align-items: center;
                    margin-bottom: 4px;
                ">
                    <div style="
                        font-size: 11px;
                        font-weight: 700;
                        text-transform: uppercase;
                        color: #444444;
                    ">Resumo — Baseline (sem fine-tuning)</div>
                    {badge(rouge_base, df_base)}
                </div>
                <div style="
                    background: #bbdefb;
                    border-radius: 4px;
                    padding: 10px;
                    font-size: 13px;
                    line-height: 1.6;
                    color: #0d47a1;
                    white-space: pre-wrap;
                ">{row_base["resumo_gerado"]}</div>
            </div>

            <div>
                <div style="
                    display: flex;
                    justify-content: space-between;
                    align-items: center;
                    margin-bottom: 4px;
                ">
                    <div style="
                        font-size: 11px;
                        font-weight: 700;
                        text-transform: uppercase;
                        color: #444444;
                    ">Resumo — Fine-Tuning</div>
                    {badge(rouge_ft, df_ft)}
                </div>
                <div style="
                    background: #ede7f6;
                    border-radius: 4px;
                    padding: 10px;
                    font-size: 13px;
                    line-height: 1.6;
                    color: #4527a0;
                    white-space: pre-wrap;
                ">{row_ft["resumo_gerado"]}</div>
            </div>
        </div>
        """
        display(HTML(html))

In [8]:
# =======================================================================
# USO — altere os caminhos e o dataset conforme necessário
# =======================================================================

# MediaSum
visualizar_comparativo(
    caminho_baseline=path_baseline,
    caminho_ft=path_ft,
    dataset="mediasum"
)

Total de amostras: 40
Dataset: MEDIASUM
